# 02.1 Hosted ChatGPT Teacher Baseline On τ³-Bench Retail

This notebook answers one narrow question:

```text
How well does a hosted ChatGPT model perform as the teacher agent in the exact same τ³-Bench retail harness?
```

This is a baseline, not the logit-distillation teacher path. We still use local Qwen/vLLM when we need same-tokenizer probabilities or logits. Here, ChatGPT is useful because it gives us a strong hosted reference point with the same tasks, same user simulator, same environment, same tools, and same reward calculation.


## What Runs Where

```mermaid
flowchart LR
  T["τ³ retail test task"] --> U["ChatGPT user simulator"]
  U --> A["Hosted ChatGPT teacher agent"]
  A --> P["Qwen XML tool-call parser"]
  P --> E["τ³ retail environment"]
  E --> A
  E --> S["τ³ reward / success score"]
```

The same local ChatGPT subscription shim serves two roles:

```text
user simulator model -> plays the customer
teacher model        -> acts as the agent being evaluated
```

They can be different models. The notebook defaults to `CHATGPT_TEACHER_MODEL` for the teacher and `TAU_BENCH_USER_SIMULATOR_LLM` for the customer.


In [1]:
from __future__ import annotations

from collections import Counter
from typing import Any
import json
import os

from pathlib import Path
import sys

cwd = Path.cwd()
ROOT = cwd if (cwd / "common" / "config.py").exists() else cwd.parent
if not (ROOT / "common" / "config.py").exists():
    raise RuntimeError("Run this notebook from the repo root or a blog folder under the repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import config as cfg
from common import retail_eval
from common import tau_runtime
from common import teacher_backends
from common import user_simulator

paths = cfg.setup_notebook_paths(blog_dir_name="1-distilling-a-0-8b-tool-calling-agent")
ROOT = paths.root
BLOG_DIR = paths.blog_dir
DATA_DIR = paths.data_dir
OUTPUT_DIR = paths.output_dir
ENV_PATH = paths.env_path
DOTENV_LOADED = paths.dotenv_loaded

TAU_BENCH_REPO_REVISION = cfg.TAU_BENCH_REPO_REVISION
TAU_BENCH_DOMAIN = "retail"

print("Project root:", ROOT)
print("Blog dir:", BLOG_DIR)
print("Output dir:", OUTPUT_DIR)
print("Loaded .env:", DOTENV_LOADED, "from", ENV_PATH)
print("Benchmark: τ³-Bench", TAU_BENCH_DOMAIN)
print("Pinned tau2-bench revision:", TAU_BENCH_REPO_REVISION[:8])


Project root: /Users/kargarisaac/codes/personal/distillation-blogs
Blog dir: /Users/kargarisaac/codes/personal/distillation-blogs/1-distilling-a-0-8b-tool-calling-agent
Output dir: /Users/kargarisaac/codes/personal/distillation-blogs/outputs
Loaded .env: True from /Users/kargarisaac/codes/personal/distillation-blogs/.env
Benchmark: τ³-Bench retail
Pinned tau2-bench revision: c42db6cc


## Start The Shared ChatGPT Shim

The shim is local and OpenAI-compatible. τ³-Bench talks to it through LiteLLM for the simulated user. Our teacher backend talks to the same shim through `/v1/chat/completions`.

Keep the model choices in `.env` when you want reproducible runs:

```text
TAU_BENCH_USER_SIMULATOR_LLM=openai/gpt-5.4-mini
CHATGPT_TEACHER_MODEL=chatgpt/gpt-5.5
CHATGPT_TEACHER_REASONING_EFFORT=medium
```


In [2]:
user_simulator_runtime = user_simulator.start_tau_bench_user_simulator_from_env(
    existing_shim=globals().get("chatgpt_user_simulator_shim")
)
chatgpt_user_simulator_shim = user_simulator_runtime.shim
TAU_BENCH_USER_SIMULATOR_LLM = user_simulator_runtime.model
TAU_BENCH_USER_SIMULATOR_ARGS = user_simulator_runtime.args

print("User simulator model:", TAU_BENCH_USER_SIMULATOR_LLM)
print("ChatGPT subscription shim default model:", user_simulator_runtime.shim_model)
print("ChatGPT subscription shim base URL:", chatgpt_user_simulator_shim.base_url)
print("User simulator args:", user_simulator.public_user_simulator_args(TAU_BENCH_USER_SIMULATOR_ARGS))
print("User simulator API key present:", bool(TAU_BENCH_USER_SIMULATOR_ARGS.get("api_key")))


User simulator model: openai/gpt-5.4-mini
ChatGPT subscription shim default model: gpt-5.4-mini
ChatGPT subscription shim base URL: http://127.0.0.1:60335/v1
User simulator args: {'api_base': 'http://127.0.0.1:60335/v1', 'base_url': 'http://127.0.0.1:60335/v1', 'temperature': 0.0, 'num_retries': 6, 'timeout': 300}
User simulator API key present: True


## Configure The Hosted Teacher

The hosted teacher receives the rendered Qwen-style prompt as text and returns the assistant completion. The rest of the harness is unchanged: we still parse Qwen XML tool calls and execute them in τ³-Bench.

This means the baseline is fair at the harness level, but it is not a same-tokenizer logit teacher.


In [3]:
tokenizer = cfg.load_tokenizer()

chatgpt_teacher_model = os.getenv(
    "CHATGPT_TEACHER_MODEL",
    os.getenv("TEACHER_MODEL", "chatgpt/gpt-5.5"),
)
chatgpt_teacher_reasoning_effort = (
    os.getenv("CHATGPT_TEACHER_REASONING_EFFORT")
    or os.getenv("TEACHER_REASONING_EFFORT")
    or "medium"
)

teacher_config = cfg.TeacherConfig(
    provider="chatgpt_raw",
    server_base_url=chatgpt_user_simulator_shim.base_url,
    model_name=chatgpt_teacher_model,
    request_model=chatgpt_teacher_model,
    temperature=float(os.getenv("CHATGPT_TEACHER_TEMPERATURE", os.getenv("TEACHER_TEMPERATURE", "0.0"))),
    top_p=float(os.getenv("CHATGPT_TEACHER_TOP_P", os.getenv("TEACHER_TOP_P", "1.0"))),
    top_k=-1,
    reasoning_effort=chatgpt_teacher_reasoning_effort,
    request_timeout_seconds=int(
        os.getenv("CHATGPT_TEACHER_REQUEST_TIMEOUT_SECONDS", os.getenv("TEACHER_REQUEST_TIMEOUT_SECONDS", "600"))
    ),
    max_new_tokens=int(os.getenv("CHATGPT_TEACHER_MAX_NEW_TOKENS", os.getenv("TEACHER_MAX_NEW_TOKENS", str(cfg.MAX_NEW_TOKENS)))),
)

server_health = teacher_backends.teacher_server_health(teacher_config)
active_teacher_model = (server_health or {}).get("model", teacher_config.model_name)

print("Teacher provider:", teacher_config.provider)
print("Teacher shim server:", teacher_config.server_base_url)
print("Teacher request model:", teacher_config.request_model)
print("Active teacher model:", active_teacher_model)
print("Teacher reasoning effort:", teacher_config.reasoning_effort)
print("Teacher temperature:", teacher_config.temperature)
print("Teacher top_p:", teacher_config.top_p)
print("Teacher max_new_tokens:", teacher_config.max_new_tokens)
print("Teacher runtime ready:", teacher_backends.teacher_runtime_is_configured(teacher_config))


Teacher provider: chatgpt_raw
Teacher shim server: http://127.0.0.1:60335/v1
Teacher request model: chatgpt/gpt-5.5
Active teacher model: chatgpt/gpt-5.5
Teacher reasoning effort: medium
Teacher temperature: 0.0
Teacher top_p: 1.0
Teacher max_new_tokens: 2048
Teacher runtime ready: True


## Load τ³-Bench Retail Runtime And Tools

The teacher only sees the rendered conversation, policy, and tool schemas. The hidden database and reward checker stay inside τ³-Bench.


In [4]:
retail_domain = tau_runtime.load_tau_bench_retail_domain(DATA_DIR, TAU_BENCH_REPO_REVISION)
tau_bench_runtime = retail_domain.runtime
retail_tasks = tau_bench_runtime.get_tasks("test")
retail_environment = retail_domain.environment
retail_tools = retail_domain.tools
retail_tool_schema_by_name = retail_domain.tool_schema_by_name

print("`tau2` package source checkout:", tau_bench_runtime.source_checkout)
print("Held-out retail tasks:", len(retail_tasks))
print("Retail tools:", len(retail_tools))
print("First test task id:", retail_tasks[0].id)
print("First five tools:", [tool["name"] for tool in retail_tools[:5]])


`tau2` package source checkout: /Users/kargarisaac/codes/personal/distillation-blogs/data/external/tau2-bench
Held-out retail tasks: 40
Retail tools: 16
First test task id: 5
First five tools: ['calculate', 'cancel_pending_order', 'exchange_delivered_order_items', 'find_user_id_by_name_zip', 'find_user_id_by_email']


## Smoke Check The User Simulator

Before evaluating the teacher, check that the simulated customer can produce a first message for one held-out task.


In [5]:
user_simulator_check = retail_eval.check_tau_bench_retail_user_simulator(
    runtime=tau_bench_runtime,
    task=retail_tasks[0],
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    user_simulator_args=TAU_BENCH_USER_SIMULATOR_ARGS,
)

print("User simulator check passed.")
print("Task id:", user_simulator_check["task_id"])
print("Model:", user_simulator_check["model"])
print("User-side tool count:", user_simulator_check["user_side_tool_count"])
if user_simulator_check["content"] and user_simulator_check["content"].strip():
    print("First simulated user message:")
    print(user_simulator_check["content"].strip())
if user_simulator_check["tool_calls"]:
    print("First simulated user tool calls:")
    print(json.dumps(user_simulator_check["tool_calls"], indent=2, ensure_ascii=False))


User simulator check passed.
Task id: 5
Model: openai/gpt-5.4-mini
User-side tool count: 0
First simulated user message:
Hi, I’d like to exchange a couple of items from my order.


## Run Hosted Teacher Eval On Held-Out Retail Tasks

This is pass@1: one conversation per task, no retry search. The result file is separate from the local Qwen teacher file because the provider slug is `chatgpt_raw`.


In [6]:
if not teacher_backends.teacher_runtime_is_configured(teacher_config):
    raise RuntimeError("ChatGPT teacher shim is not ready. Start the shim cell above, then rerun this cell.")

TAU_BENCH_GPT_TEACHER_EVAL_LIMIT = int(os.getenv("TAU_BENCH_GPT_TEACHER_EVAL_LIMIT", str(len(retail_tasks))))
TAU_BENCH_GPT_TEACHER_EVAL_MAX_STEPS = int(os.getenv("TAU_BENCH_GPT_TEACHER_EVAL_MAX_STEPS", os.getenv("TAU_BENCH_TEACHER_EVAL_MAX_STEPS", "100")))
TAU_BENCH_GPT_TEACHER_EVAL_MAX_ERRORS = int(os.getenv("TAU_BENCH_GPT_TEACHER_EVAL_MAX_ERRORS", os.getenv("TAU_BENCH_TEACHER_EVAL_MAX_ERRORS", "10")))
TAU_BENCH_GPT_TEACHER_EVAL_SEED = int(os.getenv("TAU_BENCH_GPT_TEACHER_EVAL_SEED", os.getenv("TAU_BENCH_TEACHER_EVAL_SEED", "42")))
TAU_BENCH_MLFLOW_ENABLED = cfg.env_flag("TAU_BENCH_MLFLOW_ENABLED", True)
TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS = cfg.env_flag("TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS", True)

user_simulator_slug = cfg.filename_slug(TAU_BENCH_USER_SIMULATOR_LLM)
teacher_slug = cfg.filename_slug(active_teacher_model)
teacher_eval_output_path = OUTPUT_DIR / f"{teacher_slug}_{teacher_config.provider}_tau3_bench_retail_test_official_teacher_eval_{user_simulator_slug}.json"
teacher_eval_trace_dir = OUTPUT_DIR / "local_traces" / f"{teacher_slug}_{teacher_config.provider}_tau3_bench_retail_test_official_teacher_eval_{user_simulator_slug}"
teacher_eval_mlflow_run_name = f"{teacher_slug}_{teacher_config.provider}_tau3_retail_teacher_eval_{user_simulator_slug}"
teacher_eval_mlflow_config = cfg.MlflowConfig(
    enabled=TAU_BENCH_MLFLOW_ENABLED,
    experiment_name=os.getenv("TAU_BENCH_MLFLOW_EXPERIMENT_NAME", "distillation-blogs-tau3"),
    log_full_artifacts=TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS,
    log_spans=False,
)

teacher_eval_config = cfg.TauBenchRetailEvalConfig(
    dataset_revision=TAU_BENCH_REPO_REVISION,
    student_model_name=active_teacher_model,
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    user_simulator_args=TAU_BENCH_USER_SIMULATOR_ARGS,
    max_steps=TAU_BENCH_GPT_TEACHER_EVAL_MAX_STEPS,
    max_errors=TAU_BENCH_GPT_TEACHER_EVAL_MAX_ERRORS,
    max_new_tokens=teacher_config.max_new_tokens,
    seed=TAU_BENCH_GPT_TEACHER_EVAL_SEED,
    model_role="hosted_teacher",
)

teacher_eval_task_objects = retail_tasks[:TAU_BENCH_GPT_TEACHER_EVAL_LIMIT]
teacher_eval_runner = retail_eval.TauBenchRetailTeacherEvalRunner(
    runtime=tau_bench_runtime,
    tokenizer=tokenizer,
    qwen_tools=retail_tools,
    tool_schema_by_name=retail_tool_schema_by_name,
    teacher_config=teacher_config,
    config=teacher_eval_config,
    trace_dir=teacher_eval_trace_dir,
)

print("Official τ³-Bench retail hosted teacher eval")
print("Teacher model:", active_teacher_model)
print("Teacher backend:", teacher_config.provider, teacher_config.server_base_url)
print("Teacher reasoning effort:", teacher_config.reasoning_effort)
print("User simulator model:", TAU_BENCH_USER_SIMULATOR_LLM)
print("User simulator args:", user_simulator.public_user_simulator_args(TAU_BENCH_USER_SIMULATOR_ARGS))
print("Held-out retail tasks:", len(teacher_eval_task_objects))
print("Max steps per task:", TAU_BENCH_GPT_TEACHER_EVAL_MAX_STEPS)
print("Max tool errors per task:", TAU_BENCH_GPT_TEACHER_EVAL_MAX_ERRORS)
print("Max new tokens per agent action:", teacher_config.max_new_tokens)
print("Output path:", teacher_eval_output_path)
print("Trace dir:", teacher_eval_trace_dir)
print("MLflow enabled:", teacher_eval_mlflow_config.enabled)
print("MLflow tracking URI:", teacher_eval_mlflow_config.tracking_uri)
print("MLflow experiment:", teacher_eval_mlflow_config.experiment_name)
print("MLflow full artifacts:", teacher_eval_mlflow_config.log_full_artifacts)
print("MLflow run name:", teacher_eval_mlflow_run_name)

teacher_eval_payload = retail_eval.run_tau_bench_retail_eval_tasks(
    task_objects=teacher_eval_task_objects,
    runner=teacher_eval_runner,
    output_path=teacher_eval_output_path,
    print_progress=True,
    show_progress_bar=True,
    quiet_tau2_console=True,
    mlflow_config=teacher_eval_mlflow_config,
    mlflow_run_name=teacher_eval_mlflow_run_name,
    mlflow_tags={"tau3.notebook": "02_1_teacher_eval_gpt", "tau3.runtime": teacher_config.provider},
)

teacher_eval_rows = teacher_eval_payload["task_results"]
teacher_eval_correct = sum(1 for row in teacher_eval_rows if row["is_success"])
teacher_eval_accuracy = teacher_eval_correct / len(teacher_eval_rows) if teacher_eval_rows else 0.0
status_counts = Counter("valid" if row["is_success"] else row.get("termination_reason", "unknown") for row in teacher_eval_rows)

print("\nHosted teacher official τ³-Bench retail pass@1 accuracy:", round(teacher_eval_accuracy, 4))
print(f"Correct: {teacher_eval_correct}/{len(teacher_eval_rows)}")
print("Status counts:", dict(status_counts))
print("Saved aggregate results to:", teacher_eval_output_path)
print("Saved per-task traces to:", teacher_eval_trace_dir)
if teacher_eval_mlflow_config.enabled:
    print("MLflow run:", teacher_eval_mlflow_run_name)
    print("MLflow experiment:", teacher_eval_mlflow_config.experiment_name)


Official τ³-Bench retail hosted teacher eval
Teacher model: chatgpt/gpt-5.5
Teacher backend: chatgpt_raw http://127.0.0.1:60335/v1
Teacher reasoning effort: medium
User simulator model: openai/gpt-5.4-mini
User simulator args: {'api_base': 'http://127.0.0.1:60335/v1', 'base_url': 'http://127.0.0.1:60335/v1', 'temperature': 0.0, 'num_retries': 6, 'timeout': 300}
Held-out retail tasks: 40
Max steps per task: 100
Max tool errors per task: 10
Max new tokens per agent action: 2048
Output path: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/chatgpt_gpt_5_5_chatgpt_raw_tau3_bench_retail_test_official_teacher_eval_openai_gpt_5_4_mini.json
Trace dir: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/local_traces/chatgpt_gpt_5_5_chatgpt_raw_tau3_bench_retail_test_official_teacher_eval_openai_gpt_5_4_mini
MLflow enabled: True
MLflow tracking URI: http://127.0.0.1:5050
MLflow experiment: distillation-blogs-tau3
MLflow full artifacts: True
MLflow run name: chatgpt_gpt_5_5_c

hosted_teacher eval:  80%|########  | 32/40 [00:00<?, ?task/s]

🏃 View run chatgpt_gpt_5_5_chatgpt_raw_tau3_retail_teacher_eval_openai_gpt_5_4_mini at: http://127.0.0.1:5050/#/experiments/6/runs/0e41e1b66c8346198f473803bbb19c0e
🧪 View experiment at: http://127.0.0.1:5050/#/experiments/6

Hosted teacher official τ³-Bench retail pass@1 accuracy: 0.6
Correct: 24/40
Status counts: {'user_stop': 16, 'valid': 24}
Saved aggregate results to: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/chatgpt_gpt_5_5_chatgpt_raw_tau3_bench_retail_test_official_teacher_eval_openai_gpt_5_4_mini.json
Saved per-task traces to: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/local_traces/chatgpt_gpt_5_5_chatgpt_raw_tau3_bench_retail_test_official_teacher_eval_openai_gpt_5_4_mini
MLflow run: chatgpt_gpt_5_5_chatgpt_raw_tau3_retail_teacher_eval_openai_gpt_5_4_mini
MLflow experiment: distillation-blogs-tau3
